In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Log in with your token (from Colab secret)
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# Model name
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 4-bit quantization config (to fit on A100)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Model loaded successfully!")

In [ ]:
import torch
import time

# Reuse your model, compiled_model, tokenizer, inputs
# Longer generation + sampling (better for compile gains)
prompt = "Write a detailed 400-word essay on why scaling laws matter for AI safety."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warmup (run twice for better amortization)
print("Warmup runs...")
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
    _ = compiled_model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
torch.cuda.synchronize()

# Baseline timing
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
with torch.no_grad():
    outputs_baseline = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
end.record()
torch.cuda.synchronize()
baseline_latency = start.elapsed_time(end)
baseline_tokens = len(outputs_baseline[0]) - len(inputs.input_ids[0])
baseline_tps = baseline_tokens / (baseline_latency / 1000)

print(f"Baseline: {baseline_latency:.2f} ms | {baseline_tps:.1f} TPS")

# Compiled timing
start.record()
with torch.no_grad():
    outputs_compiled = compiled_model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
end.record()
torch.cuda.synchronize()
compiled_latency = start.elapsed_time(end)
compiled_tokens = len(outputs_compiled[0]) - len(inputs.input_ids[0])
compiled_tps = compiled_tokens / (compiled_latency / 1000)

print(f"Compiled: {compiled_latency:.2f} ms | {compiled_tps:.1f} TPS")
print(f"Speedup: {baseline_latency / compiled_latency:.2f}× | TPS gain: {compiled_tps / baseline_tps:.2f}×")